In [1]:
# -*- coding: utf-8 -*-
"""
모든 ticker에 대한 밸류에이션 분석 배치 처리
"""
from DATA.stock_invest_function import *
import mysql.connector
import pandas as pd

# DB 연결 정보
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

####################################################
# 1단계: Korea_company_valuation_ver2에서 모든 ticker 추출
####################################################

try:
    print("데이터베이스 연결 중...")
    connection = mysql.connector.connect(**db_info)
    cursor = connection.cursor()

    # 모든 ticker 추출
    query = "SELECT DISTINCT ticker FROM Korea_company_valuation_ver2"
    cursor.execute(query)
    ticker_results = cursor.fetchall()
    ticker_list = [row[0] for row in ticker_results]

    print(f"총 {len(ticker_list)}개의 ticker를 찾았습니다.")

finally:
    if 'cursor' in locals():
        cursor.close()
    if 'connection' in locals():
        connection.close()

####################################################
# 2단계: 각 ticker에 대해 분석 수행
####################################################

# 결과를 저장할 리스트
final_results = []

# ticker_list = ['A005930', 'A000660']

for i, target_ticker in enumerate(ticker_list):
    print(f"\n=== {i+1}/{len(ticker_list)} - Ticker: {target_ticker} 분석 중... ===")

    try:
        ####################################################
        # valuation 데이터 추출 및 분석
        ####################################################

        connection = mysql.connector.connect(**db_info)
        cursor = connection.cursor()

        # valuation 데이터 조회
        query = f"SELECT * FROM Korea_company_valuation_ver2 WHERE ticker = %s"
        cursor.execute(query, (target_ticker,))
        results = cursor.fetchall()
        column_names = [desc[0] for desc in cursor.description]

        if not results:
            print(f"  {target_ticker}: 데이터 없음")
            continue

        df = pd.DataFrame(results, columns=column_names)

        # valuation 데이터 필터링
        valuation_df = df[df['indicator'].str.contains('valuation', case=False, na=False)].copy()
        valuation_df['value'] = pd.to_numeric(valuation_df['value'], errors='coerce')
        valuation_df = valuation_df.dropna(subset=['value'])

        if len(valuation_df) == 0:
            print(f"  {target_ticker}: valuation 데이터 없음")
            continue

        # pivot table 생성
        pivot_df = valuation_df.pivot_table(
            index='date',
            columns='indicator',
            values='value',
            aggfunc='first'
        )

        # 평균값 계산
        averages = {}
        for col in ['lstm_valuation', 'sarima_valuation', 'prophet_valuation', 'ensemble_valuation']:
            if col in pivot_df.columns:
                averages[f'{col}_평균'] = pivot_df[col].mean()

        if not averages:
            print(f"  {target_ticker}: 평균 계산할 데이터 없음")
            continue

        avg_df = pd.DataFrame(list(averages.items()), columns=['Indicator', 'Average_Value'])

        ####################################################
        # 시가총액 데이터 추출
        ####################################################

        new_target_ticker = target_ticker

        query = f"SELECT * FROM ks_listed_company_daily_marketcap WHERE ticker = %s ORDER BY date DESC"
        cursor.execute(query, (new_target_ticker,))
        marketcap_results = cursor.fetchall()

        if not marketcap_results:
            print(f"  {target_ticker}: 시가총액 데이터 없음")
            continue

        marketcap_column_names = [desc[0] for desc in cursor.description]
        marketcap_df = pd.DataFrame(marketcap_results, columns=marketcap_column_names)

        # 시가총액 데이터만 필터링
        marketcap_only_df = marketcap_df[marketcap_df['indicator'] == '시가총액'].copy()

        if len(marketcap_only_df) == 0:
            print(f"  {target_ticker}: 시가총액 indicator 없음")
            continue

        # 최신 시가총액 데이터 추출
        if marketcap_only_df['date'].dtype == 'object':
            marketcap_only_df['date'] = pd.to_datetime(marketcap_only_df['date'])

        latest_marketcap = marketcap_only_df.sort_values('date').iloc[-1]
        latest_marketcap_df = pd.DataFrame([latest_marketcap])
        latest_marketcap_df['value'] = latest_marketcap_df['value'] / 1000

        ####################################################
        # 상승여력 측정
        ####################################################

        avg_df['ticker'] = new_target_ticker
        top2_avg_df = avg_df.nlargest(2, 'Average_Value')

        merged_df = pd.merge(top2_avg_df, latest_marketcap_df, on='ticker', how='inner')

        if len(merged_df) == 0:
            print(f"  {target_ticker}: 데이터 결합 실패")
            continue

        merged_df['ratio'] = (merged_df['Average_Value'] - merged_df['value']) / merged_df['value']

        # 결과 저장
        for _, row in merged_df.iterrows():
            final_results.append({
                'original_ticker': target_ticker,
                'ticker': row['ticker'],
                'indicator': row['Indicator'],
                'average_value': row['Average_Value'],
                'current_marketcap': row['value'],
                'upside_ratio': row['ratio'],
                'upside_percentage': row['ratio'] * 100,
                'date': row['date']
            })

        print(f"  {target_ticker}: 완료 (상위 {len(merged_df)}개 지표)")

    except Exception as e:
        print(f"  {target_ticker}: 오류 발생 - {str(e)}")
        continue

    finally:
        if 'cursor' in locals():
            cursor.close()
        if 'connection' in locals():
            connection.close()

####################################################
# 3단계: 최종 결과 DataFrame 생성
####################################################

if final_results:
    final_df = pd.DataFrame(final_results)

    print(f"\n=== 전체 분석 완료 ===")
    print(f"분석 완료된 ticker 수: {len(final_df['original_ticker'].unique())}개")
    print(f"총 결과 행 수: {len(final_df)}행")

    print(f"\n=== 상승여력 상위 10개 ===")
    top_10 = final_df.nlargest(10, 'upside_percentage')
    print(top_10[['original_ticker', 'indicator', 'upside_percentage', 'current_marketcap', 'average_value']])

    print(f"\n=== 전체 결과 미리보기 ===")
    print(final_df.head())

else:
    print("분석 결과가 없습니다.")
    final_df = pd.DataFrame()

print("\n전체 배치 분석 완료!")


데이터베이스 연결 중...
총 62개의 ticker를 찾았습니다.

=== 1/62 - Ticker: A001440 분석 중... ===
  A001440: 완료 (상위 2개 지표)

=== 2/62 - Ticker: A000500 분석 중... ===
  A000500: 완료 (상위 2개 지표)

=== 3/62 - Ticker: A004000 분석 중... ===
  A004000: 완료 (상위 2개 지표)

=== 4/62 - Ticker: A007700 분석 중... ===
  A007700: 완료 (상위 2개 지표)

=== 5/62 - Ticker: A009150 분석 중... ===
  A009150: 완료 (상위 2개 지표)

=== 6/62 - Ticker: A044820 분석 중... ===
  A044820: 완료 (상위 2개 지표)

=== 7/62 - Ticker: A033500 분석 중... ===
  A033500: 완료 (상위 2개 지표)

=== 8/62 - Ticker: A036190 분석 중... ===
  A036190: 완료 (상위 2개 지표)

=== 9/62 - Ticker: A042660 분석 중... ===
  A042660: 완료 (상위 2개 지표)

=== 10/62 - Ticker: A375500 분석 중... ===
  A375500: 완료 (상위 2개 지표)

=== 11/62 - Ticker: A060980 분석 중... ===
  A060980: 완료 (상위 2개 지표)

=== 12/62 - Ticker: A071280 분석 중... ===
  A071280: 완료 (상위 2개 지표)

=== 13/62 - Ticker: A071970 분석 중... ===
  A071970: 완료 (상위 2개 지표)

=== 14/62 - Ticker: A082740 분석 중... ===
  A082740: 완료 (상위 2개 지표)

=== 15/62 - Ticker: A086390 분석 중... ===
  A086

In [2]:
final_df

,original_ticker,ticker,indicator,average_value,current_marketcap,upside_ratio,upside_percentage,date
0,A001440,A001440,sarima_valuation_평균,3.031930e+09,4.139130e+09,-0.267496,-26.749591,2025-12-18
1,A001440,A001440,ensemble_valuation_평균,2.717586e+09,4.139130e+09,-0.343440,-34.344026,2025-12-18
2,A000500,A000500,prophet_valuation_평균,1.226786e+09,1.356540e+09,-0.095650,-9.565041,2025-12-18
3,A000500,A000500,sarima_valuation_평균,1.187107e+09,1.356540e+09,-0.124901,-12.490069,2025-12-18
4,A004000,A004000,lstm_valuation_평균,1.501142e+09,1.195830e+09,0.255314,25.531376,2025-12-18
...,...,...,...,...,...,...,...,...
77,A031980,A031980,sarima_valuation_평균,1.043224e+09,9.045420e+08,0.153317,15.331709,2025-12-18
78,A084370,A084370,sarima_valuation_평균,2.058665e+09,1.645370e+09,0.251187,25.118689,2025-12-18
79,A084370,A084370,ensemble_valuation_평균,1.580602e+09,1.645370e+09,-0.039364,-3.936379,2025-12-18
80,A010140,A010140,sarima_valuation_평균,1.808982e+10,2.094400e+10,-0.136277,-13.627671,2025-12-18


In [3]:
# -*- coding: utf-8 -*-
"""
한국 기업 밸류에이션 데이터 추출 스크립트
ticker를 입력하면 해당 기업의 모든 데이터를 추출합니다.
"""
from DATA.stock_invest_function import *
import mysql.connector
import pandas as pd


In [4]:
####################################################
# 1단게 : valuation 데이터의 모든 데이터 추출
####################################################

# DB 연결 정보
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

table_name = "Korea_company_valuation_ver2"

# 추출할 ticker 설정 (여기서 변경하세요)
target_ticker = "005930"  # 삼성전자 예시
# target_ticker = ["005930", "000660", "035420"]  # 여러 ticker를 추출할 경우

try:
    print("데이터베이스 연결 중...")

    # DB 연결
    connection = mysql.connector.connect(**db_info)
    cursor = connection.cursor()

    print(f"연결 성공! {db_info['database']} 데이터베이스에 접속했습니다.")

    # ticker가 리스트인지 문자열인지 확인
    if isinstance(target_ticker, str):
        # 단일 ticker
        query = f"SELECT * FROM {table_name} WHERE ticker = %s"
        cursor.execute(query, (target_ticker,))
        print(f"Ticker '{target_ticker}'에 대한 데이터를 조회 중...")

    else:
        # 여러 ticker
        placeholders = ", ".join(["%s"] * len(target_ticker))
        query = f"SELECT * FROM {table_name} WHERE ticker IN ({placeholders})"
        cursor.execute(query, target_ticker)
        print(f"Ticker {target_ticker}에 대한 데이터를 조회 중...")

    # 결과 가져오기
    results = cursor.fetchall()

    # 컬럼명 가져오기
    column_names = [desc[0] for desc in cursor.description]

    if results:
        # DataFrame 생성
        df = pd.DataFrame(results, columns=column_names)

        print(f"\n=== 조회 결과 ===")
        print(f"추출된 데이터: {len(df)}행, {len(df.columns)}열")
        print(f"컬럼명: {list(df.columns)}")

        # 데이터 미리보기
        print(f"\n=== 데이터 미리보기 ===")
        print(df.head())

        # 기본 정보 출력
        print(f"\n=== 데이터 정보 ===")
        print(df.info())

        # DataFrame 생성 완료
        print(f"\nDataFrame 생성이 완료되었습니다.")

    else:
        print(f"해당 ticker에 대한 데이터를 찾을 수 없습니다: {target_ticker}")

except mysql.connector.Error as e:
    print(f"MySQL 연결 오류: {e}")
    print("연결 정보를 확인해주세요.")

except Exception as e:
    print(f"오류 발생: {e}")

finally:
    # 연결 종료
    if 'cursor' in locals():
        cursor.close()
        print("커서 연결이 종료되었습니다.")
    if 'connection' in locals():
        connection.close()
        print("데이터베이스 연결이 종료되었습니다.")

print("\n스크립트 실행 완료!")

try:
    # pivot table 생성
    pivot_df = df.pivot_table(
        index='date',       # 인덱스로 사용할 컬럼
        columns='indicator', # 컬럼으로 사용할 컬럼
        values='value',     # 값으로 사용할 컬럼
        aggfunc='first'     # 중복값이 있을 경우 첫 번째 값 사용
    )

    print(pivot_df.info())

except KeyError as e:
    print(f"필요한 컬럼이 없습니다: {e}")
    print(f"현재 DataFrame의 컬럼: {list(df.columns)}")

데이터베이스 연결 중...
연결 성공! investar 데이터베이스에 접속했습니다.
Ticker '005930'에 대한 데이터를 조회 중...
해당 ticker에 대한 데이터를 찾을 수 없습니다: 005930
커서 연결이 종료되었습니다.
데이터베이스 연결이 종료되었습니다.

스크립트 실행 완료!
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 86 entries, 2020-06-30 to 2027-12-31
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   mc_ets                   18 non-null     object
 1   mc_lstm                  18 non-null     object
 2   mc_prophet               18 non-null     object
 3   mc_sarima_exog           12 non-null     object
 4   mc_sarima_noexog         18 non-null     object
 5   mc_theta                 18 non-null     object
 6   psr                      59 non-null     object
 7   psr_ETS                  25 non-null     object
 8   psr_LSTM                 25 non-null     object
 9   psr_Prophet              25 non-null     object
 10  psr_SARIMA_exog          25 non-null     object
 11  psr_SARIMA_noexog

In [5]:
####################################################
# 2단게 :valuation을 포함하는 indicator 데이터만 필터링
####################################################

print("valuation 데이터 필터링 중...")
valuation_df = df[df['indicator'].str.contains('valuation', case=False, na=False)].copy()

# value 컬럼을 숫자형으로 변환
print("데이터 타입 변환 중...")
valuation_df['value'] = pd.to_numeric(valuation_df['value'], errors='coerce')

# NaN 값 제거
valuation_df = valuation_df.dropna(subset=['value'])

print(f"필터링된 데이터: {len(valuation_df)}행")
print(f"포함된 indicator: {valuation_df['indicator'].unique()}")

# pivot table 생성
print("\nPivot Table 생성 중...")
pivot_df = valuation_df.pivot_table(
    index='date',
    columns='indicator',
    values='value',
    aggfunc='first'
)

print(f"Pivot Table 생성 완료!")
print(f"크기: {pivot_df.shape[0]}행 × {pivot_df.shape[1]}열")
print(f"날짜 범위: {pivot_df.index.min()} ~ {pivot_df.index.max()}")

print("\nPivot Table 미리보기:")
print(pivot_df.head())

# 각 ticker별 평균 계산
print("\n각 indicator별 평균값 계산 중...")

# 평균값 계산
averages = {}

if 'lstm_valuation' in pivot_df.columns:
    lstm_avg = pivot_df['lstm_valuation'].mean()
    averages['lstm_valuation_평균'] = lstm_avg
    print(f"LSTM Valuation 평균: {lstm_avg:,.2f}")

if 'sarima_valuation' in pivot_df.columns:
    sarima_avg = pivot_df['sarima_valuation'].mean()
    averages['sarima_valuation_평균'] = sarima_avg
    print(f"SARIMA Valuation 평균: {sarima_avg:,.2f}")

if 'prophet_valuation' in pivot_df.columns:
    prophet_avg = pivot_df['prophet_valuation'].mean()
    averages['prophet_valuation_평균'] = prophet_avg
    print(f"Prophet Valuation 평균: {prophet_avg:,.2f}")

if 'ensemble_valuation' in pivot_df.columns:
    ensemble_avg = pivot_df['ensemble_valuation'].mean()
    averages['ensemble_valuation_평균'] = ensemble_avg
    print(f"Ensemble Valuation 평균: {ensemble_avg:,.2f}")

# 평균값을 DataFrame으로 정리
print("\n=== 평균값 요약 ===")
avg_df = pd.DataFrame(list(averages.items()), columns=['Indicator', 'Average_Value'])
print(avg_df)

# 전체 통계 정보
print("\n=== 전체 통계 정보 ===")
print(pivot_df.describe())

valuation 데이터 필터링 중...
데이터 타입 변환 중...
필터링된 데이터: 0행
포함된 indicator: []

Pivot Table 생성 중...
Pivot Table 생성 완료!
크기: 0행 × 0열
날짜 범위: NaT ~ NaT

Pivot Table 미리보기:
Empty DataFrame
Columns: []
Index: []

각 indicator별 평균값 계산 중...

=== 평균값 요약 ===
Empty DataFrame
Columns: [Indicator, Average_Value]
Index: []

=== 전체 통계 정보 ===


ValueError: Cannot describe a DataFrame without columns

In [31]:
####################################################
# 3단게 : 시가총액 데이터의 추출
####################################################

"""
ks_listed_company_daily_marketcap 테이블에서 ticker 기반 데이터 추출
"""

table_name = "ks_listed_company_daily_marketcap"

# 추출할 ticker 설정
# target_ticker = ["005930", "000660", "035420"]  # 여러 ticker를 추출할 경우

# A를 붙인 ticker 생성
if isinstance(target_ticker, str):
    new_target_ticker = "A" + target_ticker
else:
    new_target_ticker = ["A" + ticker for ticker in target_ticker]

try:
    print("데이터베이스 연결 중...")

    # DB 연결
    connection = mysql.connector.connect(**db_info)
    cursor = connection.cursor()

    print(f"연결 성공! {db_info['database']} 데이터베이스에 접속했습니다.")

    # ticker가 리스트인지 문자열인지 확인
    if isinstance(new_target_ticker, str):
        # 단일 ticker
        query = f"SELECT * FROM {table_name} WHERE ticker = %s ORDER BY date DESC"
        cursor.execute(query, (new_target_ticker,))  # new_target_ticker 사용
        print(f"Ticker '{new_target_ticker}'에 대한 시가총액 데이터를 조회 중...")

    else:
        # 여러 ticker
        placeholders = ", ".join(["%s"] * len(new_target_ticker))
        query = f"SELECT * FROM {table_name} WHERE ticker IN ({placeholders}) ORDER BY ticker, date DESC"
        cursor.execute(query, new_target_ticker)  # new_target_ticker 사용
        print(f"Ticker {new_target_ticker}에 대한 시가총액 데이터를 조회 중...")

    # 결과 가져오기
    results = cursor.fetchall()

    # 컬럼명 가져오기
    column_names = [desc[0] for desc in cursor.description]

    if results:
        # DataFrame 생성
        marketcap_df = pd.DataFrame(results, columns=column_names)

        print(f"\n=== 조회 결과 ===")
        print(f"추출된 데이터: {len(marketcap_df)}행, {len(marketcap_df.columns)}열")
        print(f"컬럼명: {list(marketcap_df.columns)}")

        # 날짜 범위 확인
        if 'date' in marketcap_df.columns:
            print(f"데이터 기간: {marketcap_df['date'].min()} ~ {marketcap_df['date'].max()}")

    else:
        print(f"해당 ticker에 대한 데이터를 찾을 수 없습니다: {new_target_ticker}")
        print("데이터베이스에서 실제 ticker 형식을 확인해보세요.")

except mysql.connector.Error as e:
    print(f"MySQL 연결 오류: {e}")
    print("연결 정보를 확인해주세요.")

except Exception as e:
    print(f"오류 발생: {e}")

finally:
    # 연결 종료
    if 'cursor' in locals():
        cursor.close()
        print("커서 연결이 종료되었습니다.")
    if 'connection' in locals():
        connection.close()
        print("데이터베이스 연결이 종료되었습니다.")

# 시가총액 데이터만 필터링
marketcap_only_df = marketcap_df[marketcap_df['indicator'] == '시가총액'].copy()

if len(marketcap_only_df) > 0:
    # date 컬럼을 datetime으로 변환 (필요시)
    if marketcap_only_df['date'].dtype == 'object':
        marketcap_only_df['date'] = pd.to_datetime(marketcap_only_df['date'])

    # 날짜순으로 정렬하고 가장 마지막(최신) 데이터 추출
    latest_marketcap = marketcap_only_df.sort_values('date').iloc[-1]

    # DataFrame 형태로도 생성
    latest_marketcap_df = pd.DataFrame([latest_marketcap])

else:
    latest_marketcap = None
    latest_marketcap_df = pd.DataFrame()


데이터베이스 연결 중...
연결 성공! investar 데이터베이스에 접속했습니다.
Ticker 'A005930'에 대한 시가총액 데이터를 조회 중...

=== 조회 결과 ===
추출된 데이터: 18725행, 4열
컬럼명: ['date', 'ticker', 'indicator', 'value']
데이터 기간: 2015-01-01 ~ 2025-09-08

=== 데이터 미리보기 ===
         date   ticker indicator         value
0  2025-09-08  A005930        종가  7.010000e+04
1  2025-09-08  A005930      시가총액  4.149670e+14
2  2025-09-08  A005930       거래량  9.263140e+06
3  2025-09-08  A005930      거래대금  6.485250e+11
4  2025-09-08  A005930     유통주식수  5.919640e+09
5  2025-09-05  A005930        종가  6.950000e+04
6  2025-09-05  A005930      시가총액  4.114150e+14
7  2025-09-05  A005930       거래량  1.152670e+07
8  2025-09-05  A005930      거래대금  8.046420e+11
9  2025-09-05  A005930     유통주식수  5.919640e+09

=== 데이터 정보 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18725 entries, 0 to 18724
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   date       18725 non-null  object 
 1   ticker     18725 

In [42]:
####################################################
# 4단게 : 상승여력 측정
####################################################

latest_marketcap_df['value'] = latest_marketcap_df['value']/1000

# 1. avg_df에 ticker 칼럼 추가
avg_df['ticker'] = new_target_ticker

# 2. Average_Value 상위 2개만 추출
top2_avg_df = avg_df.nlargest(2, 'Average_Value')

# 3. top2_avg_df와 latest_marketcap_df를 ticker 기준으로 결합
merged_df = pd.merge(top2_avg_df, latest_marketcap_df, on='ticker', how='inner')

# 4. (Average_Value - value) / value 값 계산
merged_df['ratio'] = (merged_df['Average_Value'] - merged_df['value']) / merged_df['value']

# 결과 확인
print("=== 최종 측정 DataFrame ===")
print(merged_df)

print(f"\n=== 비율 계산 결과 ===")
for idx, row in merged_df.iterrows():
    print(f"{row['Indicator']}: {row['ratio']:.4f} ({row['ratio']*100:.2f}%)")

=== 최종 측정 DataFrame ===
               Indicator  Average_Value   ticker       date indicator  \
0      lstm_valuation_평균   4.368958e+11  A005930 2025-09-08      시가총액   
1  ensemble_valuation_평균   4.208491e+11  A005930 2025-09-08      시가총액   

          value     ratio  
0  4.149670e+11  0.052845  
1  4.149670e+11  0.014175  

=== 비율 계산 결과 ===
lstm_valuation_평균: 0.0528 (5.28%)
ensemble_valuation_평균: 0.0142 (1.42%)


In [46]:
valuation_df

,date,ticker,indicator,value
222,2025-09-30,A214150,ensemble_valuation,3.752473e+09
226,2025-09-30,A214150,lstm_valuation,3.619673e+09
229,2025-09-30,A214150,prophet_valuation,4.150403e+09
232,2025-09-30,A214150,sarima_valuation,3.487343e+09
235,2025-10-31,A214150,ensemble_valuation,4.100337e+09
237,2025-10-31,A214150,lstm_valuation,3.906182e+09
240,2025-10-31,A214150,prophet_valuation,4.524175e+09
241,2025-10-31,A214150,sarima_valuation,3.870653e+09
242,2025-11-30,A214150,ensemble_valuation,4.129683e+09
244,2025-11-30,A214150,lstm_valuation,3.863438e+09
